In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# RAVE Multilingual Training - Google Colab\n",
    "\n",
    "Train RAVE on multilingual voice data (English, Kurdish, Greek, German)\n",
    "\n",
    "**GPU**: T4 (free tier)\n",
    "**Duration**: ~3-4 hours\n",
    "**Output**: rave_output.mp3 (4-5 min generated audio)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Mount Google Drive & Upload Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Mount Google Drive\n",
    "from google.colab import drive\n",
    "drive.mount('/content/drive')\n",
    "\n",
    "print(\"✅ Google Drive mounted at /content/drive\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "**TODO:** Upload `training_data.wav` to your Google Drive, then create symlink:"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import shutil\n",
    "\n",
    "# Link to your uploaded file\n",
    "drive_training_data = '/content/drive/My Drive/training_data.wav'\n",
    "\n",
    "# Create workspace\n",
    "os.makedirs('/content/rave_workspace/data', exist_ok=True)\n",
    "\n",
    "# Copy to workspace\n",
    "if os.path.exists(drive_training_data):\n",
    "    shutil.copy(drive_training_data, '/content/rave_workspace/data/training_data.wav')\n",
    "    print(f\"✅ Copied training data ({os.path.getsize('/content/rave_workspace/data/training_data.wav')/1e6:.1f} MB)\")\n",
    "else:\n",
    "    print(\"❌ training_data.wav not found in Google Drive root\")\n",
    "    print(\"   Upload it to: https://drive.google.com/ root directory\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Install Dependencies"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!pip install -q torch librosa julius soundfile scipy matplotlib"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Clone RAVE Repository"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "os.chdir('/content')\n",
    "\n",
    "!git clone https://github.com/acids-ircam/rave.git\n",
    "os.chdir('/content/rave')\n",
    "\n",
    "print(\"✅ RAVE cloned\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Verify GPU"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch\n",
    "\n",
    "print(f\"GPU Available: {torch.cuda.is_available()}\")\n",
    "if torch.cuda.is_available():\n",
    "    print(f\"GPU Name: {torch.cuda.get_device_name(0)}\")\n",
    "    print(f\"GPU Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB\")\n",
    "else:\n",
    "    print(\"⚠️ No GPU available!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Start Training"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "os.chdir('/content/rave')\n",
    "\n",
    "# Training command\n",
    "!python -m rave train \\\n",
    "  --dataset /content/rave_workspace/data \\\n",
    "  --num_epochs 100 \\\n",
    "  --batch_size 8 \\\n",
    "  --num_workers 4 \\\n",
    "  --device cuda \\\n",
    "  --output_dir /content/rave_workspace/checkpoints"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Generate Audio Output"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch\n",
    "import soundfile as sf\n",
    "import numpy as np\n",
    "from pathlib import Path\n",
    "\n",
    "# Find best checkpoint\n",
    "checkpoint_dir = Path('/content/rave_workspace/checkpoints')\n",
    "checkpoints = sorted(checkpoint_dir.glob('*.pt'))\n",
    "\n",
    "if not checkpoints:\n",
    "    print(\"❌ No checkpoints found!\")\n",
    "else:\n",
    "    best_ckpt = checkpoints[-1]  # Last (best) checkpoint\n",
    "    print(f\"Loading: {best_ckpt.name}\")\n",
    "    \n",
    "    # Load model\n",
    "    model = torch.load(best_ckpt, map_location='cuda')\n",
    "    model.eval()\n",
    "    \n",
    "    # Generate 4 minutes of audio\n",
    "    with torch.no_grad():\n",
    "        # Random latent code\n",
    "        latent = torch.randn(1, 128, 16000 * 4, device='cuda')  # 4 min\n",
    "        audio = model.decode(latent)\n",
    "    \n",
    "    # Save\n",
    "    audio_np = audio.cpu().numpy()[0]\n",
    "    output_path = '/content/rave_workspace/rave_output.wav'\n",
    "    sf.write(output_path, audio_np, 16000)\n",
    "    \n",
    "    print(f\"✅ Generated: {output_path}\")\n",
    "    print(f\"   Duration: {len(audio_np) / 16000 / 60:.1f} minutes\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Download Results"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from google.colab import files\n",
    "\n",
    "# Download generated audio\n",
    "files.download('/content/rave_workspace/rave_output.wav')\n",
    "print(\"✅ Audio downloaded to your computer\")\n",
    "\n",
    "# Download checkpoint\n",
    "files.download(str(checkpoints[-1]))\n",
    "print(\"✅ Checkpoint downloaded\")"
   ]
  }
 ],
 "metadata": {
  "accelerator": "GPU",
  "colab": {
   "provenance": []
  },
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}
